# 01 — Texas extraction imputation (Colab)

**Task:** Texas doesn't meter most pumpage (rule of capture). We learn pumpage from Kansas + Nebraska (where it *is* metered) and transfer the model to Texas. Target: `pumpage_af` (acre-feet/year per county).

**Models wired up:** XGBoost, LightGBM, CatBoost, and a stacking ensemble. All log to MLflow via `aquiferwatch.mlflow_utils.start_run` so everything we run is visible to the teammate.

---

### Where the research value actually is (read before grinding on hyperparams)

GBDT libraries (XGB/LGB/CatBoost) use the same underlying algorithm with different engineering choices. On a tabular problem this size (~3.6k rows) they typically land within a few % of each other, and stacking them usually adds another 2–5%. So the *framework choice* is the least interesting research question.

Higher-leverage experiments, in order:

1. **Spatial CV is the only honest evaluation.** Random-row CV overstates performance because rows from the same county share hidden state (geology, well network). Every comparison below uses `GroupKFold` on FIPS. If a tweak looks great under random CV but flat under spatial CV, it's overfitting to county identity — don't ship it.
2. **Feature engineering > framework swap.** Worth trying: crop-mix interactions with saturated thickness, rolling 3-year precip, well-density quantile bins, neighbor-county features (e.g. mean pumpage of adjacent counties in t-1). Each of these can move MAE more than CatBoost-vs-XGBoost will.
3. **Uncertainty bands matter as much as point accuracy.** A p50 point forecast with nothing around it is useless for policy. p10/p90 coverage near 0.80 is the acceptance bar.
4. **Transferability from KS/NE → TX is the real question.** The held-out set should be TX-like, not random counties. For now (synthetic data) spatial CV is a proxy; once real data lands, train on KS+NE only and test on TX (after reconciling against the partial TX meters that do exist).
5. **Foundation models for tabular:** *TabPFN v2* is worth a 30-min try — it's the only foundation model that's genuinely competitive on small-n tabular. Agri-specific vision foundation models (Presto, Prithvi, SatMAE) produce pixel embeddings from satellite imagery; integrating them is a 2+ day rabbit hole we shouldn't chase during the hackathon unless we have time post-accuracy-gate.

### Anti-patterns

- **Don't trust synthetic-data leaderboards.** The data generator in `aquiferwatch.data.synthetic` has known structure. Any model that approximates that structure will win. Use synthetic only to debug pipelines. Pick the final model on real data.
- **Don't tune on test.** Every metric below is on a *held-out* set; hyperparameter search must use a separate CV split of the training set. `optuna` + spatial CV is the right pattern when we're ready.
- **Don't stack three near-identical GBDTs expecting miracles.** If you want real ensemble diversity, mix a GBDT with a linear model or KNN on the residuals.


## 0. Colab bootstrap

In [ ]:
# Install aquiferwatch WITHOUT its transitive deps so we don't downgrade
# Colab's pre-installed IPython / jupyter (known to break %load_ext autoreload).
# Colab already provides numpy, pandas, pyarrow, scipy, sklearn, matplotlib.
!pip install -q --no-deps git+https://github.com/Vigneshwarr3/FusionHack_2026.git
!pip install -q lightgbm xgboost catboost mlflow boto3 pydantic-settings


In [ ]:
from aquiferwatch.colab import bootstrap

# Change team_member to your name — it becomes a tag on every MLflow run.
bootstrap(team_member="raj")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aquiferwatch.analytics.models import (
    BoostedRegressor,
    QuantileBoostedRegressor,
    StackingEnsemble,
    train_and_log,
    spatial_cv_scores,
    summarize_cv,
)

# Toggle the data source here. `True` pulls real county-level features + labels
# from `baseline.parquet` (filtered to `data_quality == 'modeled_high'`, ~248
# counties with measured thickness + composed pumping). This is cross-sectional
# (single reference year) until KS Open Records Request delivers time-panel
# pumpage. `False` uses the synthetic TX generator.
USE_REAL_DATA = True

if USE_REAL_DATA:
    from aquiferwatch.data.real import (
        build_extraction_dataset,
        spatial_train_test_split,
        REAL_EXTRACTION_FEATURE_COLS as FEATURE_COLS,
        REAL_EXTRACTION_TARGET_COL as TARGET_COL,
    )
else:
    from aquiferwatch.data.synthetic import (
        generate_tx_extraction_dataset,
        spatial_train_test_split,
        FEATURE_COLS,
        TARGET_COL,
    )


## 1. Load data

Set `USE_REAL_DATA = True` in the imports cell above (default) to pull the
real cross-sectional dataset from `baseline.parquet` — the 248 counties
where both saturated thickness (measured) and pumping (composed from crop
mix × IWMS rates) are known. That's the Tier 2 GBDT imputation training set.

Note: this is **not** a time panel yet. Panel requires KS Open Records
Request data (filed 2026-04-18) to land. Cross-sectional training is still
useful — it lets us impute pumping for the 358 `modeled_low` counties where
thickness is uncertain but crop mix is known.

Features dropped vs. synthetic contract: `precip_mm` (PRISM ingest deferred).
`well_density` is replaced by `n_wells` (we have well count per county but
not per-km²).


In [ ]:
if USE_REAL_DATA:
    df = build_extraction_dataset(quality_filter='modeled_high')
    print(f'real cross-sectional: {len(df)} rows × {df["state"].nunique()} states')
else:
    df = generate_tx_extraction_dataset(n_counties=60, n_years=20, seed=42)
    print(f'synthetic panel: {len(df)} rows')

print(df.shape)
df.head()


In [ ]:
# Quick sanity check of the target distribution.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET_COL].hist(bins=40, ax=axes[0])
axes[0].set_title("pumpage_af distribution")
df.groupby("year")[TARGET_COL].mean().plot(ax=axes[1])
axes[1].set_title("mean pumpage per year")
plt.tight_layout()
plt.show()

## 2. Spatial train/test split

Held-out *counties*, not rows. A county that appears in train should never appear in test.

In [ ]:
train_df, test_df = spatial_train_test_split(df, test_frac=0.2, seed=42)

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[TARGET_COL]
groups_train = train_df["fips"]

print(f"train rows={len(X_train)}  test rows={len(X_test)}")
print(f"train counties={train_df['fips'].nunique()}  test counties={test_df['fips'].nunique()}")

## 3. Baseline — LightGBM

Matches the pattern in `Agricultural_Data_Analysis/backend/models/yield_model.py` (single LightGBM regressor). This is our baseline. Every other model must clear it to be worth the complexity.

Edit `params` to experiment.

In [ ]:
lgb_params = {
    "n_estimators": 600,
    "num_leaves": 63,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "verbose": -1,
}

lgb_model = BoostedRegressor("lightgbm", params=lgb_params)
_ = train_and_log(
    lgb_model, X_train, y_train, X_test, y_test,
    module="extraction_imputation", run_name="tx_lightgbm_baseline",
)

## 4. XGBoost

In [ ]:
xgb_params = {
    "n_estimators": 600,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "min_child_weight": 3,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "tree_method": "hist",
}

xgb_model = BoostedRegressor("xgboost", params=xgb_params)
_ = train_and_log(
    xgb_model, X_train, y_train, X_test, y_test,
    module="extraction_imputation", run_name="tx_xgboost_baseline",
)

## 5. CatBoost

In [ ]:
cat_params = {
    "iterations": 1500,
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3.0,
    "subsample": 0.85,
    "bootstrap_type": "Bernoulli",
    "verbose": 0,
}

cat_model = BoostedRegressor("catboost", params=cat_params)
_ = train_and_log(
    cat_model, X_train, y_train, X_test, y_test,
    module="extraction_imputation", run_name="tx_catboost_baseline",
)

## 6. Stacking ensemble

Out-of-fold predictions from the three base learners → Ridge meta-learner. OOF folds are **grouped by FIPS** (see `StackingEnsemble.fit(..., groups=...)`) so no county leaks into its own meta-training row.

Try `meta="mean"` to sanity-check whether the Ridge meta is actually pulling its weight.

In [ ]:
stack = StackingEnsemble(
    base=[
        BoostedRegressor("lightgbm", params=lgb_params),
        BoostedRegressor("xgboost",  params=xgb_params),
        BoostedRegressor("catboost", params=cat_params),
    ],
    meta="ridge",
    n_folds=5,
)

_ = train_and_log(
    stack, X_train, y_train, X_test, y_test,
    module="extraction_imputation", run_name="tx_stack_lgb_xgb_cat_ridge",
    groups_train=groups_train,
)

## 7. Uncertainty bands — conformal quantile regression

Raw LightGBM quantile loss gave us p10/p90 coverage of 0.35 on the last run —
the bands were ~half as wide as the nominal 80% target. That's a known
miscalibration of boosted quantile objectives.

`ConformalQuantileRegressor` fixes this with split-conformal prediction
(Romano et al. 2019). Algorithm:

1. Hold out 30% of training as a calibration set.
2. Fit p10 + p90 quantile regressors on the remaining 70%.
3. On calibration, compute one-sided nonconformity scores
   `s_i = max(p10(x_i) - y_i, y_i - p90(x_i))`.
4. Inflate bands by `q_hat = quantile(s, 1 - α)` at inference.

The finite-sample correction gives **guaranteed marginal coverage ≥ 1 − α**.
Target is 0.80 — anything in 0.78–0.82 passes the acceptance gate.


In [ ]:
from aquiferwatch.analytics.models import ConformalQuantileRegressor

conf_params = {'n_estimators': 600, 'num_leaves': 63, 'learning_rate': 0.05, 'verbose': -1}
conf_model = ConformalQuantileRegressor(
    framework='lightgbm',
    params=conf_params,
    alpha_lo=0.1,
    alpha_hi=0.9,
    coverage_target=0.80,
    cal_frac=0.3,
)
_ = train_and_log(
    conf_model, X_train, y_train, X_test, y_test,
    module='extraction_imputation', run_name='tx_conformal_quantile_lgb',
)


## 8. Spatial CV — the honest comparison

Single held-out split is noisy with 60 counties. 5-fold GroupKFold on FIPS gives us a stable estimate + a variance signal. **A model with lower mean MAE but higher fold-to-fold std is not obviously better.**

In [ ]:
# Run spatial CV on the TRAINING set only — test set stays clean for final comparison.
X_cv, y_cv, g_cv = X_train, y_train, groups_train

factories = {
    "lightgbm": lambda: BoostedRegressor("lightgbm", params=lgb_params),
    "xgboost":  lambda: BoostedRegressor("xgboost",  params=xgb_params),
    "catboost": lambda: BoostedRegressor("catboost", params=cat_params),
}

summary_rows = []
for name, factory in factories.items():
    cv = spatial_cv_scores(factory, X_cv, y_cv, groups=g_cv, n_folds=5)
    s = summarize_cv(cv)
    s.name = name
    summary_rows.append(s)

summary = pd.concat(summary_rows, axis=1).T
summary[["mae_mean", "mae_std", "rmse_mean", "rmse_std", "r2_mean", "r2_std"]].round(3)

## 9. Scratchpad — foundation-model comparison (optional)

If there's time after we clear the accuracy gate, try `TabPFN v2`:

```python
# !pip install tabpfn
# from tabpfn import TabPFNRegressor
# tab = TabPFNRegressor(device="cuda" if torch.cuda.is_available() else "cpu")
# tab.fit(X_train, y_train)
# evaluate(y_test, tab.predict(X_test))
```

TabPFN caps at ~10k rows and ~500 features, which is fine for our county-level data.  Skip Presto / Prithvi / SatMAE (satellite foundation models) — they return pixel embeddings, not county-level features, and the integration is a multi-day project.

## 10. Decision log

Fill in after running. Don't commit this notebook with outputs (nbstripout is installed via `notebooks/README.md`).

- **Best single model:** _e.g. lightgbm (mae_mean=X.XX, mae_std=X.XX)_
- **Stacking helps?:** _yes/no, by how much on mae_mean_
- **Coverage p10–p90:** _X.XX — passes 0.78–0.82 band?_
- **Open questions / next experiments:** _…_
